In [50]:
models = ["Qwen2_5_7B","Gemma3_12B","InternVL2_5_8B"]#InternVL3_5_8B"]"Qwen2_5_7B",
concept_vector_labels = ['clean','avgprobe']#,'clean','probe','probepca','centroid']

#### Import and utilities

In [51]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from ast import literal_eval # to get dict/list from csv
import pickle
import hydra
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
plt.rcParams['figure.constrained_layout.use'] = True 

import pyrootutils,sys
pyrootutils.setup_root('.',cwd=True,pythonpath=False)
sys.path.append('./src')

In [52]:
hydra.core.global_hydra.GlobalHydra.instance().clear()
hydra.initialize("config")
cfg = hydra.compose("activations_gen", overrides=[dict(qwen='+experiment=bal_qwen7b',
                                                       intern='+experiment=bal_internvl8b',
                                                       intern3='+experiment=bal_internvl3_8b',
                                                       gemma='+experiment=bal_gemma12b')['qwen'],
                                                  ])
probelabel = 'mmp' #layer used to read hidden activations (defined in config)

/tmp/ipykernel_271225/1643727456.py:2: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  hydra.initialize("config")


In [53]:
colors=cfg.dataset.COLORS.copy()
shapes=cfg.dataset.SHAPES.copy()
marker_shapes = ["\U000025A0","\U000025B2","\U000025CF","\U00002B1F","\U00002605","\U00002764"]

def simmatrix(V):
  'returns a matrix of cosine similarities'
  n=V.shape[0]
  sims = torch.empty(n,n)
  for i in range(n):
    sims[i]=torch.cosine_similarity(V,V[i])
  return sims

In [54]:
for model_name in models:
  datapath = os.path.join('.','outputs','temp_eval',model_name) 
  for templabel in concept_vector_labels:
    print(model_name,templabel)
    m=1. #multiplier, never used
    probt_all= []
    probd_all= []
    probtrued_all = []
    df = pd.DataFrame()
    for col in colors[:]:
      for sha in shapes[:]:
        #if col =='blue' and "Qwen" in model_name:
          # hs_cont_pmna.append(torch.nan)
         # continue
        if templabel=='clean':
          with open(os.path.join(datapath,f'{col}{sha}_{templabel}.pkl'),'rb') as f:
            outputs_steer = pickle.load(f)
        else:
          with open(os.path.join(datapath,f'steer_{col}{sha}_{templabel}_{m:.2f}.pkl'),'rb') as f:
            outputs_steer = pickle.load(f)
        probt_all.append(outputs_steer['probs_t'])
        probd_all.append(outputs_steer['probs_d'])
        probtrued_all.append(outputs_steer['probs_trued'])

    probt_all = torch.cat(probt_all,dim=0).flatten(end_dim=1)
    probd_all = torch.cat(probd_all,dim=0).flatten(end_dim=1)
    probtrued_all = torch.cat(probtrued_all,dim=0).flatten(end_dim=1)

    odds = torch.arange(1,probt_all.shape[0]+1)%2==1
    probA = torch.where(odds.unsqueeze(1).repeat(1,6),probt_all,probd_all)
    probB = torch.where((~odds).unsqueeze(1).repeat(1,6),probt_all,probd_all)
    probC = probtrued_all

    metadata = []
    colnoblue = colors.copy()
    #if "Qwen" in model_name: colnoblue.remove('blue')
    basedf=pd.DataFrame()
    for i,col in enumerate(colnoblue):
      for j,sha in enumerate(shapes):
        metapd = pd.read_csv(os.path.join(datapath,f'{col}{sha}_metadata.csv'), 
                                  dtype={'ID': 'string'}, 
                                  converters={'distractors': literal_eval}
                      )
        metadict = metapd.to_dict('records')
        basedf = pd.concat([basedf,metapd])
        for n,meta in enumerate(metadict):
          metadata.append(dict(cA = meta['target_color'],
                              sA = meta['target_shape'],
                              cB = meta['dis_color'],
                              sB = meta['dis_shape'],
                              cC = meta['distractors'][-1]['color'],
                              sC = meta['distractors'][-1]['shape'],
                              #correct = td_corr[i*5+j,n].item(),
                              P_interfere= meta['P_interfere'],
                              N_distractors=meta['N_distractors'],
                              #  pref=meta['pref']
                              ))
          metadata.append(dict(cA = meta['dis_color'],
                              sA = meta['dis_shape'],
                              cB = meta['target_color'],
                              sB = meta['target_shape'],
                              cC = meta['distractors'][-1]['color'],
                              sC = meta['distractors'][-1]['shape'],
                              #correct = dt_corr[i*5+j,n].item(),
                              P_interfere= meta['P_interfere'],
                              N_distractors=meta['N_distractors'],
                              #  pref=meta['pref']
                              ))
    df = pd.DataFrame(metadata)
    df.insert(0,'ID', np.arange(df.shape[0])//2)
    df['yesA']=probA[:,:3].max(axis=1)[0]
    df['noA']=probA[:,3:].max(axis=1)[0]
    df['removal'] = df['yesA'].values<df['noA'].values
    df['yesB']=probB[:,:3].max(axis=1)[0]
    df['noB']=probB[:,3:].max(axis=1)[0]
    df['insertion'] = df['yesB'].values>df['noB'].values
    df['yesC']=probC[:,:3].max(axis=1)[0]
    df['noC']=probC[:,3:].max(axis=1)[0]
    df['control'] = df['yesC'].values>df['noC'].values
    df['all'] = df[['removal','insertion','control']].values.prod(axis=1,dtype=bool)
    df['removal'].value_counts(normalize=True), df['insertion'].value_counts(normalize=True),df['control'].value_counts(normalize=True)
    
    df.to_csv(os.path.join('.','outputs',
                         'temp_eval',
                         model_name+'_'+templabel+'.csv'),index=False)
    
  basedf['ID']=np.arange(basedf.shape[0])
  basedf.to_csv(os.path.join('.','outputs',
                        'temp_eval',
                         model_name+'_dataset.csv'),index=False)



Qwen2_5_7B clean
Qwen2_5_7B avgprobe
Gemma3_12B clean
Gemma3_12B avgprobe
InternVL2_5_8B clean
InternVL2_5_8B avgprobe


In [55]:
df

,ID,cA,sA,cB,sB,cC,sC,P_interfere,N_distractors,yesA,noA,removal,yesB,noB,insertion,yesC,noC,control,all
0,0,red,square,green,star,orange,circle,0.25,4,0.894317,0.105038,False,0.006640,0.992912,False,0.941171,0.058427,True,False
1,0,green,star,red,square,orange,circle,0.25,4,0.988384,0.010433,False,0.029430,0.970134,False,0.989760,0.009244,True,False
2,1,red,square,purple,triangle,green,pentagon,0.25,4,0.856444,0.143275,False,0.007739,0.992112,False,0.954274,0.045505,True,False
3,1,purple,triangle,red,square,green,pentagon,0.25,4,0.995912,0.003929,False,0.035085,0.964671,False,0.961638,0.038029,True,False
4,2,red,square,green,heart,orange,circle,0.50,4,0.454409,0.543754,True,0.004631,0.994561,False,0.986025,0.012814,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1291,645,orange,triangle,purple,heart,yellow,star,0.50,40,0.337720,0.660961,True,0.150923,0.848172,False,0.986951,0.012613,True,False
1292,646,purple,heart,orange,pentagon,red,square,0.75,40,0.950700,0.049043,False,0.008285,0.991393,False,0.623757,0.375540,True,False
1293,646,orange,pentagon,purple,heart,red,square,0.75,40,0.786716,0.212448,False,0.075549,0.924079,False,0.710784,0.288498,True,False
1294,647,purple,heart,blue,triangle,orange,pentagon,0.75,40,0.961485,0.037234,False,0.025153,0.974376,False,0.939590,0.059675,True,False


In [56]:
trueaccs = []
for model_name in models:
  datapath = os.path.join('.','outputs','temp_eval',model_name) 
  clean_df = pd.read_csv(os.path.join('.','outputs',
                         'temp_eval',
                         model_name+'_'+'clean'+'.csv'))
  true_all_clean = (~clean_df['removal'])*(~clean_df['insertion'])*(clean_df['control'])
  for templabel in concept_vector_labels[1:]:
    df = pd.read_csv(os.path.join('.','outputs',
                         'temp_eval',
                         model_name+'_'+templabel+'.csv'))
    trueaccs.append({
      'model_name': model_name,
      'templabel': templabel,
      'all': ((true_all_clean*df['all']).sum()/true_all_clean.sum()).item(),
      'removal': ((true_all_clean*df['removal']).sum()/true_all_clean.sum()).item(),
      'insertion': (true_all_clean*df['insertion']).sum()/true_all_clean.sum(),
      'control': (true_all_clean*df['control']).sum()/true_all_clean.sum(),
      'valid_steers': true_all_clean.sum().item(),
      'invalid_steers': (~true_all_clean).sum().item()
    })

In [57]:
clean_df[clean_df['removal']]

,ID,cA,sA,cB,sB,cC,sC,P_interfere,N_distractors,yesA,noA,removal,yesB,noB,insertion,yesC,noC,control,all


In [58]:
(true_all_clean).sum(),true_all_clean.shape[0]

(np.int64(1296), 1296)

In [59]:
trueaccs

[{'model_name': 'Qwen2_5_7B',
  'templabel': 'avgprobe',
  'all': 0.5354938271604939,
  'removal': 0.9814814814814815,
  'insertion': np.float64(0.5501543209876543),
  'control': np.float64(0.9891975308641975),
  'valid_steers': 1296,
  'invalid_steers': 0},
 {'model_name': 'Gemma3_12B',
  'templabel': 'avgprobe',
  'all': 0.10443037974683544,
  'removal': 0.5387658227848101,
  'insertion': np.float64(0.1985759493670886),
  'control': np.float64(0.9841772151898734),
  'valid_steers': 1264,
  'invalid_steers': 32},
 {'model_name': 'InternVL2_5_8B',
  'templabel': 'avgprobe',
  'all': 0.02854938271604938,
  'removal': 0.16049382716049382,
  'insertion': np.float64(0.10030864197530864),
  'control': np.float64(0.996141975308642),
  'valid_steers': 1296,
  'invalid_steers': 0}]

In [60]:
resdf = pd.DataFrame(trueaccs)
resdf

,model_name,templabel,all,removal,insertion,control,valid_steers,invalid_steers
0,Qwen2_5_7B,avgprobe,0.535494,0.981481,0.550154,0.989198,1296,0
1,Gemma3_12B,avgprobe,0.104430,0.538766,0.198576,0.984177,1264,32
2,InternVL2_5_8B,avgprobe,0.028549,0.160494,0.100309,0.996142,1296,0
